## 1. import

In [1]:
import numpy as np
import pandas as pd

## 2. config

In [2]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

## 3. models dict

In [3]:
# ==============================
# Load OOF / PRED
# ==============================
model_dict = {
    "018cat_pairwise_te_bias_from_shared_min_proba": (
        np.load(CFG.NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
    ),
    "024xgb_digit_orderedte_min_biased": (
        np.load(CFG.NPY_PATH + "oof_xgb_digit_orderedte_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_xgb_digit_orderedte_min_proba_biased.npy"),
    ),
    "025lgb_digit_te_min_biased": (
        np.load(CFG.NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"),
    ),
    "026realmlp_pair_te_min_biased": (
        np.load(CFG.NPY_PATH + "oof_realmlp_pair_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_realmlp_pair_te_min_proba_biased.npy"),
    ),
}

target_name = "018cat_pairwise_te_bias_from_shared_min_proba"

oof_t, pred_t = model_dict[target_name]

# shared の列順を base 側に合わせる
perm = [2, 0, 1]

model_dict[target_name] = (
    oof_t[:, perm],
    pred_t[:, perm],
)

target_name = "024xgb_digit_orderedte_min_biased"

oof_t, pred_t = model_dict[target_name]

# shared の列順を base 側に合わせる
perm = [2, 0, 1]

model_dict[target_name] = (
    oof_t[:, perm],
    pred_t[:, perm],
)

target_name = "025lgb_digit_te_min_biased"

oof_t, pred_t = model_dict[target_name]

# shared の列順を base 側に合わせる
perm = [2, 0, 1]

model_dict[target_name] = (
    oof_t[:, perm],
    pred_t[:, perm],
)

target_name = "026realmlp_pair_te_min_biased"

oof_t, pred_t = model_dict[target_name]

# shared の列順を base 側に合わせる
perm = [2, 0, 1]

model_dict[target_name] = (
    oof_t[:, perm],
    pred_t[:, perm],
)

## 4. show correlations

In [4]:
# ==============================
# Helpers
# ==============================
def flatten_multiclass_proba(arr: np.ndarray) -> np.ndarray:
    """
    (n_samples, n_classes) -> 1d flatten
    相関確認用。まずは全クラスまとめて見る。
    """
    if arr.ndim == 1:
        return arr.astype(np.float64)
    return arr.reshape(-1).astype(np.float64)

def rank01_pd(x: np.ndarray) -> np.ndarray:
    s = pd.Series(x)
    r = s.rank(method="average").values.astype(np.float64)
    denom = max(len(r) - 1, 1)
    return (r - 1.0) / denom

# ==============================
# Build DataFrames
# ==============================
names = list(model_dict.keys())

oof_flat = [flatten_multiclass_proba(model_dict[k][0]) for k in names]
pred_flat = [flatten_multiclass_proba(model_dict[k][1]) for k in names]

oof_lens = [len(x) for x in oof_flat]
pred_lens = [len(x) for x in pred_flat]

if len(set(oof_lens)) != 1:
    raise ValueError(f"OOF lengths mismatch: {dict(zip(names, oof_lens))}")
if len(set(pred_lens)) != 1:
    raise ValueError(f"PRED lengths mismatch: {dict(zip(names, pred_lens))}")

df_oof = pd.DataFrame(np.stack(oof_flat, axis=1), columns=names)
df_pred = pd.DataFrame(np.stack(pred_flat, axis=1), columns=names)

print("df_oof shape :", df_oof.shape)
print("df_pred shape:", df_pred.shape)

# ==============================
# Raw correlation
# ==============================
print("\n=== OOF Pearson corr (raw) ===")
display(df_oof.corr())

print("\n=== TEST Pearson corr (raw) ===")
display(df_pred.corr())

# ==============================
# Rank correlation
# ==============================
df_oof_rank = df_oof.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_oof_rank.columns = names

df_pred_rank = df_pred.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_pred_rank.columns = names

print("\n=== OOF corr (rank -> Pearson) ===")
display(df_oof_rank.corr())

print("\n=== TEST corr (rank -> Pearson) ===")
display(df_pred_rank.corr())

# ==============================
# Focused view
# ==============================
target = "025lgb_digit_te_min_biased"
base_models = [model for model in model_dict.keys() if model != target]

print(f"\n=== Focused correlations vs {target} ===")
for bm in base_models:
    print(
        f"{bm:30s} | "
        f"OOF raw={df_oof[target].corr(df_oof[bm]):.6f} | "
        f"TEST raw={df_pred[target].corr(df_pred[bm]):.6f} | "
        f"OOF rank={df_oof_rank[target].corr(df_oof_rank[bm]):.6f} | "
        f"TEST rank={df_pred_rank[target].corr(df_pred_rank[bm]):.6f}"
    )

df_oof shape : (1890000, 4)
df_pred shape: (810000, 4)

=== OOF Pearson corr (raw) ===


,018cat_pairwise_te_bias_from_shared_min_proba,024xgb_digit_orderedte_min_biased,025lgb_digit_te_min_biased,026realmlp_pair_te_min_biased
018cat_pairwise_te_bias_from_shared_min_proba,1.000000,0.995078,0.994775,0.994960
024xgb_digit_orderedte_min_biased,0.995078,1.000000,0.998641,0.995699
025lgb_digit_te_min_biased,0.994775,0.998641,1.000000,0.995700
026realmlp_pair_te_min_biased,0.994960,0.995699,0.995700,1.000000



=== TEST Pearson corr (raw) ===


,018cat_pairwise_te_bias_from_shared_min_proba,024xgb_digit_orderedte_min_biased,025lgb_digit_te_min_biased,026realmlp_pair_te_min_biased
018cat_pairwise_te_bias_from_shared_min_proba,1.000000,0.996178,0.995935,0.995901
024xgb_digit_orderedte_min_biased,0.996178,1.000000,0.999317,0.997025
025lgb_digit_te_min_biased,0.995935,0.999317,1.000000,0.997172
026realmlp_pair_te_min_biased,0.995901,0.997025,0.997172,1.000000



=== OOF corr (rank -> Pearson) ===


,018cat_pairwise_te_bias_from_shared_min_proba,024xgb_digit_orderedte_min_biased,025lgb_digit_te_min_biased,026realmlp_pair_te_min_biased
018cat_pairwise_te_bias_from_shared_min_proba,1.000000,0.942156,0.945456,0.911784
024xgb_digit_orderedte_min_biased,0.942156,1.000000,0.992057,0.908159
025lgb_digit_te_min_biased,0.945456,0.992057,1.000000,0.909005
026realmlp_pair_te_min_biased,0.911784,0.908159,0.909005,1.000000



=== TEST corr (rank -> Pearson) ===


,018cat_pairwise_te_bias_from_shared_min_proba,024xgb_digit_orderedte_min_biased,025lgb_digit_te_min_biased,026realmlp_pair_te_min_biased
018cat_pairwise_te_bias_from_shared_min_proba,1.000000,0.945409,0.952764,0.952204
024xgb_digit_orderedte_min_biased,0.945409,1.000000,0.996804,0.928774
025lgb_digit_te_min_biased,0.952764,0.996804,1.000000,0.937726
026realmlp_pair_te_min_biased,0.952204,0.928774,0.937726,1.000000



=== Focused correlations vs 025lgb_digit_te_min_biased ===
018cat_pairwise_te_bias_from_shared_min_proba | OOF raw=0.994775 | TEST raw=0.995935 | OOF rank=0.945456 | TEST rank=0.952764
024xgb_digit_orderedte_min_biased | OOF raw=0.998641 | TEST raw=0.999317 | OOF rank=0.992057 | TEST rank=0.996804
026realmlp_pair_te_min_biased  | OOF raw=0.995700 | TEST raw=0.997172 | OOF rank=0.909005 | TEST rank=0.937726


In [5]:
# 3x3 の classwise cross-corr を見る
for bm in base_models:
    print(f"\n=== {bm} vs {target} : OOF classwise cross-corr ===")
    a = model_dict[bm][0]   # base model OOF
    b = model_dict[target][0]  # target model OOF

    mat = np.zeros((a.shape[1], b.shape[1]))
    for i in range(a.shape[1]):
        for j in range(b.shape[1]):
            mat[i, j] = np.corrcoef(a[:, i], b[:, j])[0, 1]

    display(pd.DataFrame(
        mat,
        index=[f"{bm}_class{i}" for i in range(a.shape[1])],
        columns=[f"{target}_class{j}" for j in range(b.shape[1])]
    ))


=== 018cat_pairwise_te_bias_from_shared_min_proba vs 025lgb_digit_te_min_biased : OOF classwise cross-corr ===


,025lgb_digit_te_min_biased_class0,025lgb_digit_te_min_biased_class1,025lgb_digit_te_min_biased_class2
018cat_pairwise_te_bias_from_shared_min_proba_class0,0.971902,-0.254501,-0.123702
018cat_pairwise_te_bias_from_shared_min_proba_class1,-0.287913,0.996963,-0.921963
018cat_pairwise_te_bias_from_shared_min_proba_class2,-0.086618,-0.920920,0.992550



=== 024xgb_digit_orderedte_min_biased vs 025lgb_digit_te_min_biased : OOF classwise cross-corr ===


,025lgb_digit_te_min_biased_class0,025lgb_digit_te_min_biased_class1,025lgb_digit_te_min_biased_class2
024xgb_digit_orderedte_min_biased_class0,0.990794,-0.272963,-0.112047
024xgb_digit_orderedte_min_biased_class1,-0.290260,0.999520,-0.923685
024xgb_digit_orderedte_min_biased_class2,-0.087583,-0.925831,0.998044



=== 026realmlp_pair_te_min_biased vs 025lgb_digit_te_min_biased : OOF classwise cross-corr ===


,025lgb_digit_te_min_biased_class0,025lgb_digit_te_min_biased_class1,025lgb_digit_te_min_biased_class2
026realmlp_pair_te_min_biased_class0,0.973837,-0.293665,-0.083737
026realmlp_pair_te_min_biased_class1,-0.289798,0.998112,-0.922406
026realmlp_pair_te_min_biased_class2,-0.091969,-0.919956,0.993686


In [6]:
# 行和を確認する
for name, (oof, pred) in model_dict.items():
    print(name)
    print("  oof shape:", oof.shape)
    print("  pred shape:", pred.shape)
    print("  oof row sum head:", np.round(oof[:5].sum(axis=1), 6))
    print("  pred row sum head:", np.round(pred[:5].sum(axis=1), 6))
    print("  oof min/max:", float(oof.min()), float(oof.max()))
    print()

018cat_pairwise_te_bias_from_shared_min_proba
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 1.9344165225501327e-16 1.0

024xgb_digit_orderedte_min_biased
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 3.1933023818319686e-11 0.9999999093325029

025lgb_digit_te_min_biased
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 4.440628106685934e-11 0.9999999066650522

026realmlp_pair_te_min_biased
  oof shape: (630000, 3)
  pred shape: (270000, 3)
  oof row sum head: [1. 1. 1. 1. 1.]
  pred row sum head: [1. 1. 1. 1. 1.]
  oof min/max: 2.7523189244835606e-05 0.9999186461608099

